# Binary grain vs background (SwinV2 + UPerNet, 5-fold CV)

This notebook is separate from `colab_run_pipeline_221.ipynb`. It trains **binary** segmentation:

- **Class 0 (non-grain / background):** original multiclass label `0`.
- **Class 1 (grain):** any original label in `1`–`15` (bivalves, micrite, cement, …, brachiopod).
- **Ignore:** original value `255` stays ignored in the loss.

Multiclass masks are converted **on the fly** in the dataset (no duplicate mask files written).

Training uses the same **SwinV2-Tiny + UPerNet** setup as the multiclass pipeline (`num_labels=2`), **5-fold** cross-validation over image/mask pairs, and logs **pixel accuracy**, **mean IoU**, and **per-class IoU** each validation epoch. Summary metrics are saved to `cv_summary.json` under `--output_dir`.

**Typical order:** install → mount Drive → **git pull** → set paths → smoke / training cells.

In [ ]:
# Install dependencies (Colab)
!pip -q install --upgrade transformers tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2b) Pull from GitHub (full git log: stash, fetch, fast-forward, file stats — same style as the SSL/Seg Colab)
import os
from pathlib import Path

REPO = Path("/content/drive/My Drive/Payne_lab_swin_transformer")
if not REPO.is_dir():
    raise FileNotFoundError(f"Clone not found: {REPO} — mount Drive and fix the path.")
os.chdir(REPO)
print(REPO.resolve(), "\n")
!git status -sb
!git stash push -m "colab-local-before-pull"
!git pull origin feat/swin-upernet-training-pipeline

In [ ]:
import os
from pathlib import Path

# Repo on Drive (update if your clone lives elsewhere)
REPO_ROOT = Path("/content/drive/My Drive/Payne_lab_swin_transformer")

# Labeled image/mask dirs (same as colab_run_pipeline_221: adjust to your dataset folder)
IMG_DIR = "/content/drive/My Drive/Petrographic images_ML work/labelled images_PS/labelled images_PS/my_dataset/img"
MASK_DIR = "/content/drive/My Drive/Petrographic images_ML work/labelled images_PS/labelled images_PS/my_dataset/masks_machine"

# Output roots (binary CV vs multiclass/SSL in the other notebook)
OUT_ROOT_221 = "/content/drive/My Drive/Petrographic images_ML work/model_outputs_221"
OUT_BINARY = "/content/drive/My Drive/Petrographic images_ML work/model_outputs_binary_cv"

# Export for `!python ... $IMG_DIR` cells (bash reads these; same pattern as the other Colab)
os.environ["REPO_ROOT"] = str(REPO_ROOT)
os.environ["IMG_DIR"] = IMG_DIR
os.environ["MASK_DIR"] = MASK_DIR
os.environ["OUT_BINARY"] = OUT_BINARY
os.environ["OUT_ROOT_221"] = OUT_ROOT_221

print("Repo exists:", REPO_ROOT.exists())
print("IMG dir exists:", Path(IMG_DIR).exists())
print("MASK dir exists:", Path(MASK_DIR).exists())

## Smoke test

Builds dataloaders and checks paired samples (no training).

In [ ]:
# Smoke test: dataloader only (unbuffered; logs like colab_run_pipeline_221)
import os
os.chdir(REPO_ROOT)
!python -u code/model_training_pipeline/swin_binary_segmentation_221.py \
  --img_dir "$IMG_DIR" \
  --mask_dir "$MASK_DIR" \
  --output_dir "$OUT_BINARY/smoke" \
  --no_train

## Full 5-fold cross-validation

Run the **config** cell first so `REPO_ROOT` and `os.environ` (`IMG_DIR`, `MASK_DIR`, `OUT_BINARY`, `OUT_ROOT_221`) are set.

The next cell uses **`os.chdir` + `!python -u`** (same style as `colab_run_pipeline_221` segmentation) so training logs stream like the multiclass finetune: `[config]`, `[dataset]`, per-fold `Train | Val` counts, `epoch … | train_avg_loss`, `Epoch … | acc | mIoU`, two binary per-class IoU lines, and `saved …/best_upernet_swinv2_binary.pth`. Edit or remove `--backbone_checkpoint` to point at your SSL weights (default: `$OUT_ROOT_221/ssl_full/ssl_swinv2_best.pth`).

In [ ]:
# Full binary CV (same invocation style as colab_run_pipeline_221 cell 6)
import os
os.chdir(REPO_ROOT)
!python -u code/model_training_pipeline/swin_binary_segmentation_221.py \
  --img_dir "$IMG_DIR" \
  --mask_dir "$MASK_DIR" \
  --output_dir "$OUT_BINARY/cv5" \
  --n_folds 5 \
  --epochs 20 \
  --batch_size 2 \
  --crop 512 \
  --num_workers 2 \
  --backbone_checkpoint "$OUT_ROOT_221/ssl_full/ssl_swinv2_best.pth"